In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path

from azure.ai.ml import MLClient, load_job
from azure.identity import DeviceCodeCredential

# Hardcoded repo root and pipeline YAML.
ROOT = Path(r"C:\\Users\\nkz3kor\\Documents\\traffic_vlm\\production\\qwen_auto_qc")
YAML_PATH = ROOT / "configs" / "azureml_pipeline_job.yaml"
INFERENCE_SCRIPT_PATH = ROOT / "scripts" / "azure" / "run_pipeline_inference.py"

# Hardcoded Azure workspace identifiers.
SUBSCRIPTION_ID = "6d35e354-c39e-4f09-8a30-2d71bc4c833e"
RESOURCE_GROUP = "rg-autoqc-sandbox-dev"
WORKSPACE_NAME = "ml-autoqc-sandbox-dev"

# Single hardcoded inference mode.
MODE = "without_red_rectangle"  # change this one value only

t = INFERENCE_SCRIPT_PATH.read_text(encoding="utf-8")
y = YAML_PATH.read_text(encoding="utf-8")
assert "inference_input_diagnostics" in t
assert 'parser.add_argument("--inference-mode")' in t
assert "--inference-mode ${{inputs.inference_mode}}" in y

cred = DeviceCodeCredential(tenant_id="organizations")
ml_client = MLClient(
    cred,
    SUBSCRIPTION_ID,
    RESOURCE_GROUP,
    WORKSPACE_NAME,
)

job = load_job(str(YAML_PATH))
job.inputs["inference_mode"] = MODE
ts = datetime.utcnow().strftime("%Y%m%d-%H%M%S")
job.display_name = f"qwen-auto-qc-dag-{MODE}-{ts}"
created = ml_client.jobs.create_or_update(job)
print(f"SUBMITTED: {created.name} | {job.display_name} | mode={MODE}")
